## Illustration du fonctionnement de DPM Tools

Ce Notebook détaille le calcul des fonctionnelles de Minkowski, et explicite les choix faits par DPM Tools.
https://github.com/digital-porous-media/dpm_tools

### Les marching squares

Comme expliqué dans Armstong (2019), il s’agit pour chaque pixel de découper un bloc de 2×2 pixels voisins. 

Le code ci-dessous se trouve dans
dpm_tools > dpm_tools > metrics > _minkowski_utils.py

On parcourt chaque pixel (x, y) de l'image. On regarde la valeur du pixel d'ancrage (x, y) (l.10), puis celle du voisin à droite (l.11), puis du voisin en dessous (l.12), puis du voisin en bas à droite (l.13). Les valeurs 1 ou 0 de ces pixels sont pondérées par 1, 2, 4, 8 respectivement, puis additionnées pour former un nombre binaire à quatre chiffres. Au total, $2^4 = 16$ possibilités.

In [ ]:
@jit('u8[:](u1[:, :], u8, u8)', nopython=True, parallel=True)
def get_configs_histogram_2d(image, dim0, dim1):
    n_threads = numba.get_num_threads()
    thread_hist = np.zeros((n_threads, 6), dtype=np.uint64)
    IC = initialize_mapping(2)

    for x in prange(dim0 - 1):
        thread_id = numba.get_thread_id()
        for y in range(dim1 - 1):
            mask_val = (int(image[x, y] == 1) +
                        int(image[x + 1, y] == 1) * 2 +
                        int(image[x, y + 1] == 1) * 4 +
                        int(image[x + 1, y + 1] == 1) * 8)
            thread_hist[thread_id, IC[mask_val]] += 1

    config_hist = np.sum(thread_hist, axis=0)

    return config_hist

Ces $2^4$ = 16 combinaisons de couleurs possibles pour chaque bloc sont attribuables aux six cas présentés ci-dessous par rotation des blocs. Dans DPM tools, le code binaire à 4 chiffres obtenu précédemment est attribué à un des 6 cas par le vecteur IC.

![alt text](configurations.png "Configurations")

In [ ]:
IC = np.array([0, 1, 1, 2, 1, 2, 3, 4, 1, 3, 2, 4, 2, 4, 4, 5])

A la fin (l.16 et 18), on incrémente l'histogramme des cas $A_0$, $A_1$, etc.

NB : en réalité, on ne balaye pas le pixel d'ancrage (x, y) sur l'ensemble de l'image. On exclut les bordures droite et inférieure, afin de pouvoir regarder le voisinage (l.11-13). Cela créée des effets de bords sur lesquels nous reviendrons.

### Reconstruction des fonctionnelles de Minkowski.

L'histogramme construit à partir des marching squares nous permet de remonter aux fonctionnelles de Minkowski, grâce aux formules données dans Armstrong (2019) :

$$
\begin{array}{rcl}
M_0 &=& (N_{A_1} + 2 N_{A_2} + 2 N_{A_3} + 3 N_{A_4} + 4 N_{A_5}) / 4 \\
M_1 &=& (2 N_{A_1} + 2 N_{A_2} + 4 N_{A_3} + 2 N_{A_4}) / 2 \\
M_2 &=& \frac{1}{2\pi}(\frac{\pi}{2} N_{A_1} + 2 \frac{\pi}{2} N_{A_3} - \frac{\pi}{2} N_{A_4})  = (N_{A_1} + 2 N_{A_3} - N_{A_4}) / 4
\end{array}
$$

Il s'agit pour chaque square de calculer la surface de nuage ($M_0$), la longueur d'interface interne au square ($M_1$), et le rayon de courbure moyen normalisé par $2\pi$ ($M_2$). Puis, corriger par 4 pour $M_0$, car chaque pixel est compté 4 fois ; par 2 pour $M_1$, car chaque interface est comptée deux fois.

Un exemple pour s'entrainer à calculer les MF par l'algorithme de Armstrong (2019). Attention, par rapport à l'image, la définition des cas $A_i$ est changée.

![alt text](exemple.png "Exemple")

Un problème : il y a un biais de numérisation, où on approxime l'interface courbe par des pixels. Mantz (2008) a ainsi montré que les fonctionnelles de Minkowski appliquées à une image de pixels surestiment le périmètre d'un disque d'un facteur $4/\pi$.

![alt text](mantz_2008.png "Mantz 2008")

Un deuxième problème : On ne sait pas vraiment, dans le cas $A_3$, si c'est une discrétisation d'une zone nuageuse déconnectée deux fois concave ($2\times \pi / 2$), ou connectée deux fois convexe ($2 \times -\pi / 2$). Selon qu'on choisisse si l'intersection des quatre pixels est noire ou blanche, respectivement.

Aussi DPM Tools résout le premier problème en ajoutant un facteur $\pi/4$ par rapport à la formule de Armstong (2019) (l.22), et en moyennant le décompte du cas $A_3$ dans $M_2$ entre la 4-connectedness et la 8-connectedness (l.4-5, l.23-25).

Code issu de :

dpm_tools > dpm_tools > metrics > _minkowski_coeff.py

dpm_tools > dpm_tools > metrics > _scalars.py


In [ ]:
contributions_2d = {
    "v2": np.array([0, 1, 2, 2, 3, 4]),
    "v1": np.array([0, 2, 2, 4, 2, 0]),
    "v0_4": np.array([0, 1, 0, 2, -1, 0]),  # 4-connected
    "v0_8": np.array([0, 1, 0, -2, -1, 0])
}

def _minkowski_2d(image: np.ndarray) -> Tuple[float, float, float]:
    """
    Helper function to compute the 2D Minkowski functionals (area, perimeter, Euler Characteristic)

    Parameters:
        image: The binary image where the phase of interest is 1.
    Returns:
        Tuple[float, float, float]: Area, perimeter, radius of curvature
    """

    # Get the isotropic configurations (3D)
    nx, ny = image.shape
    configs_hist = get_configs_histogram_2d(image, nx, ny)
    v2 = np.sum(contributions_2d["v2"] / 4. * configs_hist)
    v1 = np.sum(contributions_2d["v1"] / 8. * np.pi * configs_hist)
    v0_8 = np.sum(contributions_2d["v0_8"] / 4. * configs_hist)
    v0_4 = np.sum(contributions_2d["v0_4"] / 4. * configs_hist)
    v0 = (v0_4 + v0_8) / 2

    return v2, v1, v0

Pour preuve, sur l'exemple précédent, on n'obtient pas la valeur attendue de $M_1 = 8$ px.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from dpm_tools.metrics import minkowski_functionals

img_exemple = np.array([[0, 0, 0, 0],
                        [0, 1, 0, 0],
                        [0, 1, 1, 0],
                        [0, 0, 0, 0]], dtype='bool')

M0, M1, M2 = minkowski_functionals(img_exemple, pad=False)

print(M0)
print(M1)
print(M2)
print(M1 * 4/np.pi)

3.0
6.283185307179586
1.0
8.0


### Effet de bord

Comme mentionné précédemment, la taille non unitaire du bloc $2\times2$ conduit à des effets de bord : le pixel d'ancrage (en haut à gauche) ne peut pas être promené sur les bordures inférieure et droite de l'image. Cela conduit à un biais sur l'estimation de $M_0$, i.e. la couverture nuageuse. En effet, les 4 coins de l'images ne seront comptés qu'une fois mais corrigés d'un facteur 1/4 (voir section "Reconstruction des fonctionnelles de Minkowski"), et les bordures haut, bas, droit, gauche ne seront comptées que deux fois, bien qu'elles aussi corrigés d'un facteur 1/4.

Dans les calculs numériques, il faut tenir compte de ces biais, sous peine de quoi la couverture nuageuse peut être sur- ou sous-estimée quand la taille de l'image (en px) devient trop petite pour négliger la contribution des bordures, voire des coins.

Dans l'exemple ci-dessous, l'axe des abscisses représente la dégradation en résolution d'une même image : la taille de l'image diminue pour les échelles les plus grossières. Les courbes colorées sont calculées avec DPM Tools. Les courbes noires continues représentent la couverture nuageuse réelle, et les courbes en pointillés sont un calcul de couverture nuageuse où on reproduit le biais de DPM tools sur les bords, pas sur les coins.

![alt text](biais_m0.png "Biais M0")

Et là, en corrigeant les biais de bords et de coins.

![alt text](biais_m0_full.png "Biais M0")

In [ ]:
self.dpm_cloud_cover = (np.sum(img[1:-1,1:-1])
                        + np.sum(img[1:-1,-1])/2
                        + np.sum(img[-1,1:-1])/2
                        + np.sum(img[0,1:-1])/2
                        + np.sum(img[1:-1,0])/2
                        + img[0,0]/4
                        + img[0,-1]/4
                        + img[-1,0]/4
                        + img[-1,-1]/4) / img.shape[0]**2

Bien que l'aire de nuage $M_0$ soit très simple à calculer numériquement, et sans la méthode des marching squares, il est possible d'apporter une correction statistique aux effets de bord. En notant $N$ le côté de l'image carrée, on compte $4$ coins, $4(N-2)$ pixels de bordure, et $(N-2)^2$ pixels intérieurs. En notant $c$ la probabilité (supposée homogène) qu'un pixel soit nuageux, on a :

$$ \left\{
\begin{array}{rcl}
    M_0^\mathrm{vrai} &=& 4c + 4(N-2)c + (N-2)^2c = N^2 c \\
    M_0^\mathrm{dpm} &=& \frac{1}{4}4c + \frac{1}{2}4(N-2)c + (N-2)^2c \\
\end{array}
\right. $$

Ce qui donne la correction suivante :

$$ M_0^\mathrm{vrai} = \left(\frac{N}{N-1}\right)^2\,M_0^\mathrm{dpm}$$

Une telle correction statistique repose sur l'hypothèse que la couverture nuageuse $c$ est homogène sur l'ensemble de l'image. L'image ci-dessous montre les MF de DPM tools, corrigées par méthode statistique (courbes colorées). Pour certaines images (courbe jaune), la correction est très efficace. Pour d'autres, la correction rapproche du vrai $m_0$ sans l'épouser (courbes violette, verte, bleue). Pour d'autres enfin, le biais DPM Tools conduisait par hasard à une meilleure mesure que sa version corrigée (courbe rose).

![alt text](corr_m0.png "Corr M0")

Un autre effet de bord concerne $M_1$. Les interfaces entre les pixels de bordure ne sont visitées qu'une fois, mais sont corrigées par un facteur 2. On compte $4(N-1)$ interfaces de bordure, et $2(N-1)(N-2)$ interfaces interieures (bel et bien visitées deux fois). En notant $p$ la probabilité (supposée homogène entre bordure et intérieur) d'avoir une interface entre deux pixels, on a :

$$ \left\{
\begin{array}{rcl}
M_1^\mathrm{vrai} &=& 4(N-1)p + 2(N-1)(N-2)p \\
M_1^\mathrm{dpm} &=& \frac{1}{2}4(N-1)p + 2(N-1)(N-2)p \\
\end{array}
\right. $$

Ce qui donne la correction suivante :
$$ M_1^\mathrm{vrai} = \frac{N}{N-1}M_1^\mathrm{dpm}$$

### References

Armstrong, Ryan T. et al. (2019). Porous Media Characterization Using Minkowski Functionals: Theories, Applications and Future Directions. Transport in Porous Media, 130, 305–335

Mantz et al. (2008). Utilizing Minkowski functionals for image analysis: a marching square algorithm.